## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, upper, lpad, regexp_replace, concat, lower, substring

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.customers")
df.display()

## Transformations

### 1. TRIM whitespace from all string columns

In [0]:
string_cols = ["customer_id", "customer_unique_id", 
                "customer_city", "customer_state"]

for c in string_cols:
    df = df.withColumn(c, trim(col(c)))

### 2. Fix customer_zip_code_prefix - pad with leading zeros to 5 digits

In [0]:
df = df.withColumn(
  "customer_zip_code_prefix", 
  lpad(col("customer_zip_code_prefix"), 5, "0")
)

### 3. Standardize customer_state - UPPERCASE (should be 2-letter)

In [0]:
df = df.withColumn(
    "customer_state",
    upper(trim(col("customer_state")))
)

### 4. Standardize customer_city - Title Case & normalize spaces

In [0]:
# Fixed
df = df.withColumn(
    "customer_city",
    trim(regexp_replace(col("customer_city"), r"\s+", " "))
)
df = df.withColumn(
    "customer_city",
    concat(
        upper(substring(col("customer_city"), 1, 1)),
        lower(substring(col("customer_city"), 2, 9999))
    )
)



### 5. Handle nulls - filter out rows missing critical keys

In [0]:
df = df.filter(
    col("customer_id").isNotNull() &
    col("customer_unique_id").isNotNull() 
    )


### 6. Remove duplicate customer_id rows

In [0]:
df = df.dropDuplicates(["customer_id"])


### 7. Preview quality check

In [0]:
print(f"Total rows after cleaning: {df.count()}")
df.filter(col("customer_state").isNull()).count()  # Should be 0 or minimal

df.display()

## Write to silver layer

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("olist.silver.customers")